In [ ]:
from models import *
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from torchvision import transforms
batch_size = 32
device = 'mps'

In [3]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), 
    transforms.ToTensor()
])

In [4]:
def test_evaluate(n, file, type='mds_net'):
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    types = {'unsorted': NeuralNetUnsorted4, 'nofc': NeuralNetNoFC4, 'nodist': NeuralNetNoDist4, 'avg': NeuralNetAvg4}
    if type == 'mds_net': model = models[n]().to(device)
    else: model = types[type]().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCEWithLogitsLoss()
    correct = 0
    total_loss = 0.0
    total = 0
    test_inputs = np.memmap(f'test_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    test_targets = np.memmap(f'test_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    dataset = TensorDataset(torch.from_numpy(test_inputs), torch.from_numpy(test_targets))
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_logits = []
    with torch.no_grad():
        for batch_inputs, batch_targets in test_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            logits = model(*inputs)
            all_logits.append(logits.squeeze().cpu().numpy())
            loss = loss_function(logits.squeeze(), batch_targets)
            total_loss += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            predictions = logits.squeeze() > 0
            correct += (predictions == batch_targets).sum().item()
    all_logits = np.concatenate(all_logits)
    roc_auc = roc_auc_score(test_targets, all_logits)
    return total_loss / total, correct / total, roc_auc

In [5]:
def bcs_evaluate(n, file):
    model = NeuralNetBCS().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCEWithLogitsLoss()
    correct = 0
    total_loss = 0.0
    total = 0
    test_inputs = np.memmap(f'test_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    test_targets = np.memmap(f'test_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    dataset = TensorDataset(torch.from_numpy(test_inputs), torch.from_numpy(test_targets))
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_logits = []
    with torch.no_grad():
        for batch_inputs, batch_targets in test_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            references, query = inputs[:-1], inputs[-1]
            references = torch.stack(references, dim=1)
            query_emb, best_ref_emb = model(references, query)
            cosine_similarity = nn.functional.cosine_similarity(query_emb, best_ref_emb)
            with open('bcs_model/calibration_params.txt', mode='r') as f:
                w, b = f.read().split()
                w, b = float(w), float(b)
                logits = w * cosine_similarity + b
            all_logits.append(logits.squeeze().cpu().numpy())
            loss = loss_function(logits.squeeze(), batch_targets)
            total_loss += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            predictions = logits.squeeze() > 0
            correct += (predictions == batch_targets).sum().item()
    all_logits = np.concatenate(all_logits)
    roc_auc = roc_auc_score(test_targets, all_logits)
    return total_loss / total, correct / total, roc_auc

In [6]:
def pes_evaluate(n, file):
    model = NeuralNet2().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCELoss()
    correct = 0
    total_loss = 0.0
    total = 0
    test_inputs = np.memmap(f'test_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    test_targets = np.memmap(f'test_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    dataset = TensorDataset(torch.from_numpy(test_inputs), torch.from_numpy(test_targets))
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for batch_inputs, batch_targets in test_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            probs = torch.zeros(batch_targets.size(0)).to(device)
            for i in range(len(inputs)-1):
                probs += F.sigmoid(model(inputs[i], inputs[-1])).squeeze()
            probs /= (len(inputs)-1)
            all_probs.append(probs.squeeze().cpu().numpy())
            loss = loss_function(probs.squeeze(), batch_targets)
            total_loss += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            predictions = probs.squeeze() > 0.5
            correct += (predictions == batch_targets).sum().item()
    all_probs = np.concatenate(all_probs)
    roc_auc = roc_auc_score(test_targets, all_probs)
    return total_loss / total, correct / total, roc_auc

In [ ]:
mds_loss2, mds_acc2, mds_roc2 = test_evaluate(2, 'model_2/epoch_14.pth')
mds_loss3, mds_acc3, mds_roc3 = test_evaluate(3, 'model_3/epoch_15.pth')
mds_loss4, mds_acc4, mds_roc4 = test_evaluate(4, 'model_4/epoch_15.pth')
mds_loss5, mds_acc5, mds_roc5 = test_evaluate(5, 'model_5/epoch_14.pth')
mds_loss6, mds_acc6, mds_roc6 = test_evaluate(6, 'model_6/epoch_15.pth')

In [82]:
bcs_loss2, bcs_acc2, bcs_roc2 = bcs_evaluate(2, 'bcs_model/epoch_15.pth')
bcs_loss3, bcs_acc3, bcs_roc3 = bcs_evaluate(3, 'bcs_model/epoch_15.pth')
bcs_loss4, bcs_acc4, bcs_roc4 = bcs_evaluate(4, 'bcs_model/epoch_15.pth')
bcs_loss5, bcs_acc5, bcs_roc5 = bcs_evaluate(5, 'bcs_model/epoch_15.pth')
bcs_loss6, bcs_acc6, bcs_roc6 = bcs_evaluate(6, 'bcs_model/epoch_15.pth')

In [83]:
pes_loss2, pes_acc2, pes_roc2 = pes_evaluate(2, 'model_2/epoch_14.pth')
pes_loss3, pes_acc3, pes_roc3 = pes_evaluate(3, 'model_2/epoch_14.pth')
pes_loss4, pes_acc4, pes_roc4 = pes_evaluate(4, 'model_2/epoch_14.pth')
pes_loss5, pes_acc5, pes_roc5 = pes_evaluate(5, 'model_2/epoch_14.pth')
pes_loss6, pes_acc6, pes_roc6 = pes_evaluate(6, 'model_2/epoch_14.pth')

In [ ]:
unsorted_loss4, unsorted_acc4, unsorted_roc4 = test_evaluate(4, 'model_4_unsorted/epoch_12.pth', type='unsorted')
nofc_loss4, nofc_acc4, nofc_roc4 = test_evaluate(4, 'model_4_nofc/epoch_14.pth', type='nofc')
nodist_loss4, nodist_acc4, nodist_roc4 = test_evaluate(4, 'model_4_nodist/epoch_15.pth', type='nodist')
avg_loss4, avg_acc4, avg_roc4 = test_evaluate(4, 'model_4_avg/epoch_13.pth', type='avg')